In [12]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from pandas.tseries.offsets import Week

In [2]:
CSV_PATH = Path(r"features_weekly.csv")

INSIDERS_CSV   = "cert_dataset/insiders/insiders.csv"

threat_files = {
    'ACM2278': 'cert_dataset/insiders/r6.2-1.csv',
    'CMP2946': 'cert_dataset/insiders/r6.2-2.csv',
    'PLJ1771': 'cert_dataset/insiders/r6.2-3.csv',
    'CDE1846': 'cert_dataset/insiders/r6.2-4.csv',
    'MBG3183': 'cert_dataset/insiders/r6.2-5.csv',
}


K              = 25          
AE_EPOCHS      = 1000
AE_BATCH       = 64
AE_LR          = 1e-3
AE_WD          = 1e-5
IF_ESTIMATORS  = 2000        
IF_MAX_SAMPLES = 2000
ALPHA_AGG      = 0.7        
BETA_ENS       = 0.5        

In [4]:
df = pd.read_csv(CSV_PATH)

df.columns = df.columns.str.strip().str.lower()

df["week_start"] = pd.to_datetime(df["week_start"], errors="coerce")


In [5]:
l = pd.read_csv(INSIDERS_CSV)
INSIDERS = set(l[l["dataset"] == 6.2]["user"].values)
print(f"Insiders : {INSIDERS}")

Insiders : {'PLJ1771', 'CDE1846', 'MBG3183', 'ACM2278', 'CMP2946'}


In [ ]:
threat_weeks = {}
for user, filepath in threat_files.items():
    tmp = pd.read_csv(filepath, header=None, on_bad_lines="skip")
    tmp[2] = pd.to_datetime(tmp[2], errors="coerce")
    week_starts = (
        tmp[2].dropna()
        .dt.to_period("W")
        .apply(lambda p: p.start_time.date())
        .unique()
    )
    threat_weeks[user] = set(week_starts)


threat_weeks

{'ACM2278': {datetime.date(2010, 8, 16), datetime.date(2010, 8, 23)},
 'CMP2946': {datetime.date(2011, 1, 31),
  datetime.date(2011, 2, 7),
  datetime.date(2011, 2, 14),
  datetime.date(2011, 2, 21),
  datetime.date(2011, 2, 28),
  datetime.date(2011, 3, 7),
  datetime.date(2011, 3, 14),
  datetime.date(2011, 3, 21),
  datetime.date(2011, 3, 28)},
 'PLJ1771': {datetime.date(2010, 8, 9), datetime.date(2010, 8, 16)},
 'CDE1846': {datetime.date(2011, 2, 21),
  datetime.date(2011, 3, 14),
  datetime.date(2011, 3, 21),
  datetime.date(2011, 3, 28),
  datetime.date(2011, 4, 4),
  datetime.date(2011, 4, 11),
  datetime.date(2011, 4, 25)},
 'MBG3183': {datetime.date(2010, 10, 11)}}

In [18]:
df = df.sort_values("week_start")
df["week"] = df["week_start"].rank(method="dense").astype(int)

df["week_period"] = df["week_start"].dt.to_period("W")

all_threat_periods = set()

for weeks in threat_weeks.values():
    for w in weeks:
        all_threat_periods.add(pd.Timestamp(w).to_period("W"))

THREAT_WEEK_INTS = df[
    df["week_period"].isin(all_threat_periods)
]["week"].unique()

print(sorted(THREAT_WEEK_INTS))

[np.int64(32), np.int64(33), np.int64(34), np.int64(41), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(67), np.int64(69)]


In [20]:
feature_cols = [
    c for c in df.columns 
    if c not in ("user", "week", "week_start", "week_period")
]

X_all = df[feature_cols].values


mask_train = (
    ~df["user"].isin(INSIDERS) &
    ~df["week"].isin(THREAT_WEEK_INTS)
).values

print(f"Lignes total : {len(df)} | Lignes training : {mask_train.sum()}")


X_train_raw = X_all[mask_train]

scaler = StandardScaler()
scaler.fit(X_train_raw)

X_scaled = scaler.transform(X_all)

X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

X_train = X_scaled[mask_train]

Lignes total : 284061 | Lignes training : 223977


In [21]:
class Autoencoder(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 6),
        )
        self.decoder = nn.Sequential(
            nn.Linear(6, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, n_features),
        )
 
    def forward(self, x):
        return self.decoder(self.encoder(x))
 


In [22]:
n_features = X_train.shape[1]
X_tensor_train = torch.FloatTensor(X_train)
X_tensor_all   = torch.FloatTensor(X_scaled)
 
dataset_ae = TensorDataset(X_tensor_train)
loader_ae  = DataLoader(dataset_ae, batch_size=AE_BATCH, shuffle=True)
 
ae_model   = Autoencoder(n_features)
optimizer  = torch.optim.Adam(ae_model.parameters(), lr=AE_LR, weight_decay=AE_WD)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=AE_EPOCHS)
criterion  = nn.MSELoss()
 
best_loss, best_state = float("inf"), None
 
for epoch in range(AE_EPOCHS):
    ae_model.train()
    epoch_loss = 0
    for (batch,) in loader_ae:
        reconstructed = ae_model(batch)
        loss = criterion(reconstructed, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    avg_loss = epoch_loss / len(loader_ae)
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_state = {k: v.clone() for k, v in ae_model.state_dict().items()}
    if epoch % 100 == 0:
        print(f"  Epoch {epoch:4d} — Loss: {avg_loss:.5f}")
 
ae_model.load_state_dict(best_state)
print(f"  Best loss: {best_loss:.5f}")
 
ae_model.eval()
with torch.no_grad():
    recon = ae_model(X_tensor_all)
    ae_errors = torch.mean((X_tensor_all - recon) ** 2, dim=1).numpy()
 

  Epoch    0 — Loss: 0.52856
  Epoch  100 — Loss: 0.28007
  Epoch  200 — Loss: 0.26823
  Epoch  300 — Loss: 0.26138
  Epoch  400 — Loss: 0.25639
  Epoch  500 — Loss: 0.25276
  Epoch  600 — Loss: 0.24906
  Epoch  700 — Loss: 0.24585
  Epoch  800 — Loss: 0.24355
  Epoch  900 — Loss: 0.24198
  Best loss: 0.24126


In [23]:
if_model = IsolationForest(
    n_estimators=IF_ESTIMATORS,
    max_samples=IF_MAX_SAMPLES,
    max_features=1.0,
    contamination="auto",
    random_state=42,
    n_jobs=-1,
)
if_model.fit(X_train)
if_scores = -if_model.score_samples(X_scaled)   
print("  Isolation Forest entraîné.")

  Isolation Forest entraîné.


In [24]:
norm = MinMaxScaler()
ae_norm = norm.fit_transform(ae_errors.reshape(-1, 1)).flatten()
if_norm = norm.fit_transform(if_scores.reshape(-1, 1)).flatten()

ensemble_score = BETA_ENS * ae_norm + (1 - BETA_ENS) * if_norm

In [25]:
df_weekly = df[["user", "week"]].copy()
df_weekly["ae_score"]       = ae_norm
df_weekly["if_score"]       = if_norm
df_weekly["ensemble_score"] = ensemble_score
 
def robust_aggregate(scores):
    """0.7 * max + 0.3 * mean — capture les pics rares ET l'activité soutenue."""
    return ALPHA_AGG * scores.max() + (1 - ALPHA_AGG) * scores.mean()
 
df_user = (
    df_weekly.groupby("user")["ensemble_score"]
    .agg(robust_aggregate)
    .reset_index()
    .rename(columns={"ensemble_score": "final_score"})
)
 
df_user_debug = df_weekly.groupby("user")["ensemble_score"].agg(
    max_score="max",
    mean_score="mean",
    std_score="std",
    n_weeks_above_90=lambda x: (x > x.quantile(0.90)).sum(),
).reset_index()
df_user = df_user.merge(df_user_debug, on="user")

In [26]:
df_user = df_user.sort_values("final_score", ascending=False).reset_index(drop=True)
df_user["rank"] = df_user.index + 1
df_user["is_insider"] = df_user["user"].isin(INSIDERS).astype(int)
 
print("\n=== RÉSULTATS ===")
print(f"\nTop-{K} users :")
print(df_user.head(K)[["rank", "user", "final_score", "max_score", "mean_score", "is_insider"]].to_string(index=False))
 
insider_rows = df_user[df_user["is_insider"] == 1][["rank", "user", "final_score", "max_score", "mean_score"]]
print(f"\nRanking des {len(INSIDERS)} insiders :")
print(insider_rows.to_string(index=False))
 
top_k_hits = (df_user[df_user["is_insider"] == 1]["rank"] <= K).sum()
mean_rank  = df_user[df_user["is_insider"] == 1]["rank"].mean()
total      = len(df_user)
 
print(f"\nTop-{K} hit rate : {top_k_hits} / {len(INSIDERS)}")
print(f"Mean rank        : {mean_rank:.1f} / {total}")
print(f"Mean rank %      : {mean_rank/total*100:.1f}%")


=== RÉSULTATS ===

Top-25 users :
 rank    user  final_score  max_score  mean_score  is_insider
    1 ACM2278     0.591186   0.761870    0.192924           1
    2 BVB1673     0.481489   0.525078    0.379780           0
    3 CHB1062     0.450378   0.482206    0.376112           0
    4 SDH2394     0.447953   0.488151    0.354156           0
    5 IMC3296     0.433899   0.473481    0.341541           0
    6 CHM1561     0.431389   0.480211    0.317472           0
    7 EPI3052     0.412289   0.433518    0.362753           0
    8 WSA1765     0.408114   0.424616    0.369610           0
    9 CAM3050     0.405955   0.432731    0.343476           0
   10 QGH0041     0.405579   0.435958    0.334694           0
   11 QRJ1597     0.403889   0.452769    0.289836           0
   12 LCP0344     0.401734   0.445766    0.298995           0
   13 BPD2437     0.399765   0.421661    0.348676           0
   14 MAB1775     0.395507   0.418063    0.342878           0
   15 WMM0873     0.390825   0.4239

In [28]:
for user in INSIDERS:
    user_weeks = df_weekly[df_weekly["user"] == user].sort_values("ensemble_score", ascending=False)
    print(f"\n{user} — top semaines anormales :")
    print(user_weeks[["week", "ae_score", "if_score", "ensemble_score"]].head(10).to_string(index=False))
          


PLJ1771 — top semaines anormales :
 week  ae_score  if_score  ensemble_score
   32  0.211164  0.675165        0.443165
   33  0.202708  0.645541        0.424124
   34  0.198503  0.632184        0.415343
   31  0.005535  0.388540        0.197038
   13  0.004455  0.378270        0.191362
   27  0.004174  0.332177        0.168176
   28  0.004158  0.317516        0.160837
   30  0.004108  0.314465        0.159286
   29  0.004170  0.313716        0.158943
    8  0.004598  0.311076        0.157837

CDE1846 — top semaines anormales :
 week  ae_score  if_score  ensemble_score
   72  0.002377  0.315849        0.159113
   71  0.002188  0.302875        0.152532
   73  0.001979  0.287219        0.144599
   69  0.003423  0.271715        0.137569
   74  0.002026  0.266073        0.134049
   67  0.004053  0.233074        0.118563
   64  0.004357  0.189610        0.096983
   68  0.001546  0.176845        0.089196
   70  0.001346  0.170793        0.086070
   66  0.001584  0.166360        0.083972

MBG